# House Price Predictor

End-to-end regression project with EDA, cleaning, three regression models, 5-fold cross-validation, feature importance, and residual analysis.

## 1. Project Setup

The notebook looks for the Kaggle House Prices `train.csv` inside `data/`. If it is not found, it attempts to download a public mirror so the notebook can still be run without paid services or private credentials.

In [ ]:
import os
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
DATA_PATH = 'data/train.csv'
DATA_URL = 'https://raw.githubusercontent.com/selva86/datasets/master/HousePrices.csv'
if not os.path.exists(DATA_PATH):
    try:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
        print('Downloaded a public House Prices dataset to data/train.csv')
    except Exception as e:
        raise FileNotFoundError('Could not download the dataset. Put the Kaggle House Prices train.csv in data/ and run again.') from e
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print('Data types:')
display(df.dtypes.value_counts())
print('Missing values (top 20):')
display(df.isna().sum().sort_values(ascending=False).head(20))
print('Summary statistics:')
display(df.describe(include='all').T.head(20))

In [ ]:
target_candidates = ['SalePrice', 'sale_price', 'price', 'Price']
target = next((c for c in target_candidates if c in df.columns), None)
if target is None:
    raise ValueError('Target column not found. Expected SalePrice (Kaggle House Prices).')
print('Target:', target)
plt.figure(figsize=(8, 5))
plt.hist(df[target].dropna(), bins=40)
plt.title('Distribution of House Prices')
plt.xlabel('Sale Price')
plt.ylabel('Number of Houses')
plt.tight_layout()
plt.savefig('outputs/target_distribution.png', dpi=150)
plt.show()

## 3. Cleaning and Feature Preparation

Numerical columns use median imputation. Categorical columns use the most frequent value followed by one-hot encoding. Preprocessing is kept inside the model pipeline to reduce data leakage.

In [ ]:
X = df.drop(columns=[target]).copy()
y = df[target].copy()
id_columns = [c for c in X.columns if c.lower() in ['id', 'index']]
if id_columns:
    X = X.drop(columns=id_columns)
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', numeric_pipeline, numeric_features), ('cat', categorical_pipeline, categorical_features)], remainder='drop')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

## 4. Train Three Regression Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
}
fitted_models = {}
test_rmse = {}
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    fitted_models[name] = pipe
test_rmse[name] = rmse
    print(f'{name}: Test RMSE = {rmse:,.2f}')

## 5. 5-Fold Cross-Validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
comparison = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='neg_root_mean_squared_error')
    cv_rmse = -scores
    comparison.append({'Model': name, 'CV RMSE Mean': cv_rmse.mean(), 'CV RMSE Std': cv_rmse.std(), 'Test RMSE': test_rmse[name]})
results = pd.DataFrame(comparison).sort_values('CV RMSE Mean').reset_index(drop=True)
results.to_csv('outputs/model_comparison.csv', index=False)
display(results.style.format({'CV RMSE Mean':'{:,.2f}','CV RMSE Std':'{:,.2f}','Test RMSE':'{:,.2f}'}))
winner_name = results.loc[0, 'Model']
winner = fitted_models[winner_name]
print('Winning model:', winner_name)

## 6. Winning Model — Feature Importance

In [ ]:
model_obj = winner.named_steps['model']
prep_obj = winner.named_steps['preprocessor']
feature_names = prep_obj.get_feature_names_out()
if hasattr(model_obj, 'feature_importances_'):
    importances = model_obj.feature_importances_
else:
    importances = np.abs(model_obj.coef_)
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values('Importance', ascending=False).head(15)
display(importance_df)
plt.figure(figsize=(9, 6))
plt.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1])
plt.title(f'Top 15 Feature Importances — {winner_name}')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Residual Analysis

In [ ]:
winner_predictions = winner.predict(X_test)
residuals = y_test - winner_predictions
plt.figure(figsize=(8, 5))
plt.scatter(winner_predictions, residuals, alpha=0.6)
plt.axhline(0, linestyle='--')
plt.title(f'Residuals vs Predicted — {winner_name}')
plt.xlabel('Predicted Price')
plt.ylabel('Residual (Actual - Predicted)')
plt.tight_layout()
plt.savefig('outputs/residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Mean residual:', residuals.mean())
print('Residual standard deviation:', residuals.std())

## 8. Why This Model Won

The winning model is selected objectively using the lowest mean 5-fold cross-validation RMSE. A lower RMSE means predictions are, on average, closer to actual sale prices. Tree-based models can capture nonlinear relationships and interactions between housing features, while basic Linear Regression is limited to linear relationships. The feature-importance and residual plots provide additional evidence about which variables drive predictions and where the model makes larger errors.

## 9. Final Deliverables

- `outputs/model_comparison.csv` — model comparison table
- `outputs/feature_importance.png` — winning-model feature importance
- `outputs/residuals.png` — residual diagnostic plot
- `outputs/target_distribution.png` — target EDA plot
